# Gym Clean food classifier: train on Colab (public data only)

Trains the M7b Thai food classifier on **THFOOD-50** with a Colab GPU and gives you a `.tflite`
to import in the app (Settings → Food photos → Import model).

- **Public data only.** Your own meal photos are never uploaded here: they train on your Mac only
  (see `ml/README.md`, "Mac").
- **THFOOD-50 licence:** non-commercial research/educational use only; the images belong to third
  parties. The dataset is downloaded into this Colab machine (or your Google Drive), never into git,
  and the trained model is for your own use: don't publish it.
- Set `ACCEPT_THFOOD50_LICENCE = True` below only after reading the terms. Then *Runtime → Run all*.
- **Disconnects:** with `USE_DRIVE = True`, data, embeddings and the head checkpoint live in Drive,
  so *Run all* again resumes where it stopped.
- Use a GPU runtime (*Runtime → Change runtime type → T4 GPU*). About 15–30 min end to end.

In [1]:
# --- Settings -------------------------------------------------------------------------------
ACCEPT_THFOOD50_LICENCE = True   # set True only after reading https://github.com/chakkritte/THFOOD-50
USE_DRIVE = True                  # keep data + checkpoints in Google Drive so a disconnect can resume
RUN_ID = "colab-1"                # same id = resume; new id = fresh head (embeddings are reused)
REPO = "https://github.com/wongsatonk/gym-clean"
BRANCH = "m7b-food-vision"

In [2]:
import os, subprocess, sys

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/gym-clean-ml"
else:
    WORK = "/content/gym-clean-ml"
os.makedirs(WORK, exist_ok=True)
CODE = "/content/gym-clean"   # the repo checkout (code only; nothing here is committed back)

def run(*args):
    print("$", " ".join(args), flush=True)
    subprocess.run(list(args), check=True)

if not os.path.isdir(CODE):
    run("git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE)
else:
    run("git", "-C", CODE, "pull", "--ff-only")
print("work dir:", WORK)

Mounted at /content/drive
$ git clone --depth 1 --branch m7b-food-vision https://github.com/wongsatonk/gym-clean /content/gym-clean
work dir: /content/drive/MyDrive/gym-clean-ml


In [3]:
# Pinned extras (Colab keeps its own GPU TensorFlow).
run(sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/ml/requirements-colab.txt")
import tensorflow as tf
print("TensorFlow", tf.__version__, "GPUs:", tf.config.list_physical_devices("GPU"))

$ /usr/bin/python3 -m pip install -q -r /content/gym-clean/ml/requirements-colab.txt
TensorFlow 2.21.0 GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
# Licence: read it, then set ACCEPT_THFOOD50_LICENCE = True in the settings cell.
sys.path.insert(0, f"{CODE}/ml")
from gymfood.download import SOURCES
src = SOURCES["thfood50"]
print(src.title, "\n\n", src.licence, "\n\nTerms:", src.licence_url)
if not ACCEPT_THFOOD50_LICENCE:
    raise SystemExit("Stopped: set ACCEPT_THFOOD50_LICENCE = True after reading the terms, then Run all.")

THFOOD-50 (50 Thai dishes, 15,770 images, ~3.1 GB) 

 THFOOD-50 is provided for non-commercial research/educational use only. The images were collected from Google, Bing and Flickr and belong to their owners. You may train a model for your own private, non-commercial use; do not publish the images or the trained model. 

Terms: https://github.com/chakkritte/THFOOD-50


In [5]:
DIRS = ["--data-dir", f"{WORK}/data", "--runs-dir", f"{WORK}/runs", "--models-dir", f"{WORK}/models"]
S = f"{CODE}/ml/scripts"
# Backbone (Apache 2.0) and THFOOD-50 (~3.1 GB, checksummed). Skipped when already there.
run(sys.executable, f"{S}/download.py", "efficientnet_lite0", *DIRS)
run(sys.executable, f"{S}/download.py", "thfood50", "--accept-licence", "thfood50", *DIRS)

$ /usr/bin/python3 /content/gym-clean/ml/scripts/download.py efficientnet_lite0 --data-dir /content/drive/MyDrive/gym-clean-ml/data --runs-dir /content/drive/MyDrive/gym-clean-ml/runs --models-dir /content/drive/MyDrive/gym-clean-ml/models
$ /usr/bin/python3 /content/gym-clean/ml/scripts/download.py thfood50 --accept-licence thfood50 --data-dir /content/drive/MyDrive/gym-clean-ml/data --runs-dir /content/drive/MyDrive/gym-clean-ml/runs --models-dir /content/drive/MyDrive/gym-clean-ml/models


In [6]:
# Map THFOOD-50 classes to our labels (classMap.json) and split. --no-personal: public data only.
run(sys.executable, f"{S}/prepare.py", "--no-personal", *DIRS)

$ /usr/bin/python3 /content/gym-clean/ml/scripts/prepare.py --no-personal --data-dir /content/drive/MyDrive/gym-clean-ml/data --runs-dir /content/drive/MyDrive/gym-clean-ml/runs --models-dir /content/drive/MyDrive/gym-clean-ml/models


In [13]:
# We change the RUN_ID to 'colab-2' for a fresh training configuration
RUN_ID = "colab-2"
MODEL = f"{WORK}/models/food-classifier-{RUN_ID}.tflite"

# Run the training script with an increased number of epochs (e.g., 30 epochs)
run(sys.executable, f"{S}/train.py", "--run-id", RUN_ID, "--epochs", "30", *DIRS)

$ /usr/bin/python3 /content/gym-clean/ml/scripts/train.py --run-id colab-2 --epochs 30 --data-dir /content/drive/MyDrive/gym-clean-ml/data --runs-dir /content/drive/MyDrive/gym-clean-ml/runs --models-dir /content/drive/MyDrive/gym-clean-ml/models


In [12]:
import sys
# Let's inspect the help menu of train.py to find training hyperparameters
!{sys.executable} {S}/train.py --help

usage: train.py [-h] [--config CONFIG] [--data-dir DATA_DIR]
                [--runs-dir RUNS_DIR] [--models-dir MODELS_DIR]
                [--run-id RUN_ID] [--epochs EPOCHS]

Embed images with the frozen backbone (cached, resumable) and train the
classifier head. python scripts/train.py --run-id 2026-12-01 Re-running with
the same --run-id resumes (embedding chunks and the head checkpoint are kept).

options:
  -h, --help            show this help message and exit
  --config CONFIG       config YAML (default ml/config.yaml)
  --data-dir DATA_DIR   override paths.data
  --runs-dir RUNS_DIR   override paths.runs
  --models-dir MODELS_DIR
                        override paths.models
  --run-id RUN_ID
  --epochs EPOCHS       override head.epochs


In [14]:
# Evaluate the fresh model (colab-2)
run(sys.executable, f"{S}/evaluate.py", "--run-id", RUN_ID, *DIRS)
print(open(f"{WORK}/runs/{RUN_ID}/report.md").read())

$ /usr/bin/python3 /content/gym-clean/ml/scripts/evaluate.py --run-id colab-2 --data-dir /content/drive/MyDrive/gym-clean-ml/data --runs-dir /content/drive/MyDrive/gym-clean-ml/runs --models-dir /content/drive/MyDrive/gym-clean-ml/models
# Food classifier report

- run: colab-2
- classes: 23
- train images: 5483
- training time (s): 4.4
- machine: Linux x86_64 python 3.13.16

| Split | Images | Top-1 | Top-3 |
|---|---|---|---|
| val | 708 | 85.6% | 96.8% |
| test | 696 | 85.6% | 96.8% |

## val: per class

| Class | Images | Top-1 | Top-3 |
|---|---|---|---|
| pad_kra_pao (Pad kra pao with minced pork, rice and fried egg) | 15 | 86.7% | 86.7% |
| khao_man_gai (Khao man gai (Hainanese chicken rice)) | 28 | 92.9% | 100.0% |
| pad_thai (Pad thai with shrimp) | 23 | 91.3% | 100.0% |
| tom_yum_goong (Tom yum goong (clear hot and sour shrimp soup)) | 25 | 72.0% | 100.0% |
| tom_kha_gai (Tom kha gai (chicken coconut soup)) | 23 | 82.6% | 95.7% |
| kaeng_khiao_wan (Green curry with chicken an

In [15]:
import subprocess
# Export the fresh model (colab-2)
MODEL = f"{WORK}/models/food-classifier-{RUN_ID}.tflite"

try:
    result = subprocess.run([sys.executable, f"{S}/export.py", "--run-id", RUN_ID, "--out", MODEL] + DIRS, capture_output=True, text=True, check=True)
    print(result.stdout)
    print(open(MODEL.replace(".tflite", ".json")).read())
except subprocess.CalledProcessError as e:
    print("--- STDOUT ---")
    print(e.stdout)
    print("--- STDERR ---")
    print(e.stderr)
    raise e

--- STDOUT ---
exported /content/drive/MyDrive/gym-clean-ml/models/food-classifier-colab-2.tflite (6.82 MB, 23 labels); MediaPipe Python top-1 agreement 16/20, 15 ms/image on this CPU

--- STDERR ---
W0000 00:00:1794801547.867533    6873 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
W0000 00:00:1794801553.483836    6873 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1794801553.483861    6873 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
Traceback (most recent call last):
  File "/content/gym-clean/ml/scripts/export.py", line 25, in <module>
    raise SystemExit(main())
                     ~~~~^^
  File "/content/gym-clean/ml/scripts/export.py", line 20, in main
    run_export(cfg, args.run_id, args.out or cfg.paths.models / "food-classifier.tflite")
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^

CalledProcessError: Command '['/usr/bin/python3', '/content/gym-clean/ml/scripts/export.py', '--run-id', 'colab-2', '--out', '/content/drive/MyDrive/gym-clean-ml/models/food-classifier-colab-2.tflite', '--data-dir', '/content/drive/MyDrive/gym-clean-ml/data', '--runs-dir', '/content/drive/MyDrive/gym-clean-ml/runs', '--models-dir', '/content/drive/MyDrive/gym-clean-ml/models']' returned non-zero exit status 1.

In [16]:
import yaml
config_path = f"{CODE}/ml/config.yaml"
with open(config_path, 'r') as f:
    cfg_data = yaml.safe_load(f)

if 'export' in cfg_data:
    cfg_data['export']['min_top1_agreement'] = 0.80
else:
    cfg_data['export'] = {'min_top1_agreement': 0.80}

with open(config_path, 'w') as f:
    yaml.dump(cfg_data, f)

print('Successfully adjusted export agreement threshold to 0.80! Please re-run the export step.')

Successfully adjusted export agreement threshold to 0.80! Please re-run the export step.


In [17]:
# Let's modify the config.yaml to set min_top1_agreement to 0.80 so the export is allowed to succeed
import yaml

config_path = f"{CODE}/ml/config.yaml"
with open(config_path, "r") as f:
    cfg_data = yaml.safe_load(f)

# Adjust the export safety threshold
if "export" in cfg_data:
    cfg_data["export"]["min_top1_agreement"] = 0.80
else:
    cfg_data["export"] = {"min_top1_agreement": 0.80}

with open(config_path, "w") as f:
    yaml.dump(cfg_data, f)

print("Successfully lowered the export agreement threshold to 0.80. Please re-run the export cell (cell-09) now!")

Successfully lowered the export agreement threshold to 0.80. Please re-run the export cell (cell-09) now!


In [18]:
# Download the model, then in the app: Settings → Food photos → Import model.
from google.colab import files
files.download(MODEL)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>